# UWCT: E004, the YOLO baseline on enhanced images (dataset v2, D-017)

**Purpose:** train the E001 baseline again on Harsh's enhanced copies of the same images, for a fair
raw vs enhanced comparison. **The only change from E001 is the images.** Model (`yolo26s.pt`), imgsz 640,
100 epochs, patience 30, batch 16, seed 0, `ultralytics==8.4.155` and the GPU (Tesla T4) all match E001.

**Data:** dataset **v2** (frozen) = the v1 sessions, split and labels byte for byte, with images taken
from the `enhanced_2026-09` source by filename. The folders Harsh delivered are *not* used as a split.
zExtra is excluded. The training data file has **no test split**.

**Code:** pinned to commit `97ac710` on `p4/colab-baseline` (PR #4). This notebook only calls
repository scripts, and all checks live in the code. Runbook: `docs/COLAB.md`.

**History:** E004 was trained with this notebook on 2026-09-30 (one launch, 81 epochs; records in
`experiments/records/E004/`). Afterwards the cells were renumbered, the output helpers moved up,
the epoch logger split from the full-run cell, and a resume cell added; the commands are unchanged.

**Rules for running it:**
- Runtime → **T4 GPU**. Every cell raises on failure; do not continue past a failed cell.
- Output is kept short on purpose (long output crashed Colab during E001): script output goes to
  log files under `/content/logs/`, and only short summaries are printed.
- Smoke test, full run and resume are each enabled by a flag you set on purpose.
- Runs, checkpoints and `run_record.json` go to `MyDrive/uwct/runs/E004/` (outside Git).
- This notebook never touches the **test split** (CLAUDE.md rule 7). The E004 test evaluation is on
  hold until D-018's review-list check is done.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 2. SETTINGS -- the only values specific to this run. Everything else comes from the E004 config.
# ---------------------------------------------------------------------------------------------
import os, re, subprocess, sys

# Code to run: the exact commit, which must be reachable from BRANCH on GitHub.
REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "p4/colab-baseline"                              # protected integration branch
COMMIT = "97ac710a2da5096e3a4a377ad2c809d7565637ad"       # PR #4 merge: the commit E004 was trained with

# Experiment: the config names the frozen dataset version (v2) and every training argument.
EXPERIMENT_CONFIG = "configs/experiments/E004_yolo_enhanced.yaml"
EXPERIMENT_ID = "E004"                                    # must match `id:` in the config (checked later)

# Name of the Colab Secret holding a read-only GitHub token (fine-grained, this repo only).
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"

# Paths. The /content/... paths are local to this Colab machine and deleted when it disconnects.
# Anything under /content/drive/ is on Google Drive and persists.
REPO_DIR = "/content/uwct"                                # fixed: resume expects the same dataset path
RUNS_ROOT = "/content/drive/MyDrive/uwct/runs"            # persistent: run folders + checkpoints
LOCAL_RAW = "/content/raw/biology"                        # local copy of the raw Biology data (labels)
WEIGHTS_DIR = "/content/weights"                          # pretrained checkpoint downloaded by Ultralytics
ENHANCED_ZIP = "/content/drive/MyDrive/uwct/raw/enhanced_2026-09.zip"  # Harsh's delivery (D-017)
LOCAL_ENHANCED = "/content/raw/enhanced_2026-09"          # local unpacked copy of the enhanced images

# A short or mistyped sha would check out the wrong code without complaint.
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "COMMIT must be the full 40-character commit sha"


def sh(*cmd):
    """Run a command inside the repo, stream its output live, and raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


print("settings OK:", EXPERIMENT_ID, "| commit", COMMIT[:7], "| branch", BRANCH)

In [ ]:
# ---------------------------------------------------------------------------------------------
# 3. OUTPUT HELPERS -- keep the notebook output short (very long output can crash Colab).
#     step(msg): one timestamped progress line.
#     run(label, *cmd): run a repo command; its FULL output goes to /content/logs/<label>.log
#                       and only the last `tail` meaningful lines are printed; raises on failure.
#                       Progress-bar lines (Ultralytics' bars, "it/s") are kept in the log but
#                       not in the printed tail, so the tail shows results, not bars.
#     Logs live on this machine only; run records and checkpoints are written to Drive by the
#     scripts themselves. Times are Colab time (UTC).
# ---------------------------------------------------------------------------------------------
import collections, time

LOG_DIR = "/content/logs"
os.makedirs(LOG_DIR, exist_ok=True)
PROGRESS_RE = re.compile(r"━|it/s|s/it|\d+%\|")  # patterns that mark a progress-bar line


def step(msg):
    """Print one short progress line with the current time."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def run(label, *cmd, tail=15):
    """Run a command in the repo; full output -> log file, last `tail` non-progress lines -> notebook."""
    log_path = os.path.join(LOG_DIR, re.sub(r"[^A-Za-z0-9_.-]+", "_", label) + ".log")
    step(f"START {label}")
    t0 = time.time()
    last = collections.deque(maxlen=tail)
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line)
            if line.strip() and not PROGRESS_RE.search(line):
                last.append(line.rstrip())
        code = p.wait()
    for line in last:
        print("    " + line[:200])  # cap very long lines too
    secs = time.time() - t0
    if code != 0:
        raise RuntimeError(f"FAILED {label} (exit {code}) after {secs:.0f}s -- full log: {log_path}")
    step(f"DONE  {label} ({secs:.0f}s) -- full log: {log_path}")


step("helpers ready (progress bars filtered)")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 4. GPU CHECK -- training must run on a GPU, and E001 ran on a Tesla T4. Using the same GPU
#    type keeps the raw vs enhanced comparison fair (the GPU is not a variable of E004).
# ---------------------------------------------------------------------------------------------
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert smi.returncode == 0 and smi.stdout.strip(), \
    "NO GPU -- Runtime -> Change runtime type -> T4 GPU, then re-run from cell 2"

gpu_name = smi.stdout.strip()
print("GPU:", gpu_name)
if "T4" not in gpu_name:
    # Not fatal, but the GPU is recorded in run_record.json; note it when comparing with E001.
    print("WARNING: E001 was trained on a Tesla T4; this run uses a different GPU.")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 5. GOOGLE DRIVE -- mount it, then check before any long step that:
#    - Harsh's enhanced ZIP is where cell 2 expects it (exact name: a doubled ".zip.zip" from a
#      manual rename is the usual cause of "not found");
#    - whether an E004 run folder exists (a new full run refuses if it does; resume needs it);
#    - there is room for the run folder (E001's folder is a good size estimate).
#    Nothing is written to Drive here except creating the runs folder if it is missing.
# ---------------------------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS_ROOT, exist_ok=True)


def folder_gb(path):
    """Total size of the files under `path`, in GB."""
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for name in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, name))
            except OSError:
                pass
    return total / 1e9


assert os.path.isfile(ENHANCED_ZIP), f"Harsh's ZIP not found at {ENHANCED_ZIP} -- upload it there first"
print("enhanced ZIP:", round(os.path.getsize(ENHANCED_ZIP) / 1e9, 2), "GB")

print("existing run folders:", sorted(os.listdir(RUNS_ROOT)))
e001_dir = os.path.join(RUNS_ROOT, "E001")
if os.path.isdir(e001_dir):
    print("E001 run folder:", round(folder_gb(e001_dir), 2), "GB  (E004 will need about the same)")

e004_dir = os.path.join(RUNS_ROOT, EXPERIMENT_ID)
print("E004 folder already exists?", os.path.exists(e004_dir))

# Free space as Colab sees the Drive mount. It can differ from the real Drive quota,
# so drive.google.com/drive/quota is the authoritative number.
st = os.statvfs("/content/drive/MyDrive")
print("Drive free space (as seen by Colab):", round(st.f_bavail * st.f_frsize / 1e9, 2), "GB")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 6. CODE -- clone the repo and check out exactly COMMIT (detached), so the run is reproducible
#    and run_record.json records the same sha.
#    Security: the GitHub token is read from Colab Secrets only inside git_remote(), and handed to
#    that one git process through GIT_ASKPASS. It never appears in a command line, URL, file,
#    .git/config or printed output (errors are redacted). In a new notebook, enable the secret
#    for it first (key icon -> Notebook access).
# ---------------------------------------------------------------------------------------------
import stat
from google.colab import userdata

REPO_URL = f"https://github.com/{REPO}.git"   # no credentials in the URL
ASKPASS = "/content/uwct_git_askpass.sh"      # helper script; holds no secret itself
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    """Local git command (no network, no credentials); returns stdout, raises on failure."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    """clone/fetch: the token exists only in this git child's environment, for this call only."""
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0",
           "UWCT_GIT_TOKEN": userdata.get(TOKEN_SECRET_NAME)}
    try:
        # credential.helper= (empty) disables every credential helper, so nothing is stored.
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env,
                           capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd="/content")
else:  # re-run after a reconnect: fetch the branch again instead of re-cloning
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)

# The run must use exactly COMMIT, from the expected repo, with no local modifications.
assert git("rev-parse", "HEAD") == COMMIT, "checked-out commit differs from COMMIT"
assert git("remote", "get-url", "origin") == REPO_URL, "unexpected remote URL"
status = git("status", "--porcelain")
assert status == "", f"working tree is not clean:\n{status}"
print("commit:", git("log", "-1", "--format=%h %s"))
print("working tree: clean")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 7. REQUIREMENTS -- install the pinned Ultralytics version from requirements-gpu.txt
#    (ultralytics==8.4.155, same as E001). torch + CUDA are NOT installed here: they come
#    preinstalled with Colab. The versions are printed and recorded in run_record.json, and
#    compared with E001's below, because Colab's preinstalled torch can change over time.
# ---------------------------------------------------------------------------------------------
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")

# Check in a fresh Python process, so it sees exactly what was installed. Ultralytics may print
# its own messages on first import, so the versions go on one line tagged "VERSIONS" and only
# that line is parsed.
E001_VERSIONS = {"ultralytics": "8.4.155", "torch": "2.11.0+cu128", "cuda": "12.8"}  # experiments/registry.csv
check = subprocess.run(
    [sys.executable, "-c",
     "import ultralytics, torch; print('VERSIONS', ultralytics.__version__, torch.__version__, "
     "torch.version.cuda, torch.cuda.is_available())"],
    capture_output=True, text=True, check=True).stdout
tagged = [line.split()[1:] for line in check.splitlines() if line.startswith("VERSIONS ")]
assert len(tagged) == 1, f"could not read the versions from:\n{check}"
ultra, torch_v, cuda_v, gpu_ok = tagged[0]
now = {"ultralytics": ultra, "torch": torch_v, "cuda": cuda_v}
print("installed:", now, "| torch sees GPU:", gpu_ok)

assert now["ultralytics"] == E001_VERSIONS["ultralytics"], "Ultralytics version differs from the pin"
assert gpu_ok == "True", "torch cannot see the GPU"
for k in ("torch", "cuda"):
    if now[k] != E001_VERSIONS[k]:
        # Not fatal (the config pins only Ultralytics), but it is a difference from E001
        # and must be mentioned when E001 and E004 are compared.
        print(f"NOTE: {k} {now[k]} differs from E001's {E001_VERSIONS[k]}")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 8. ENVIRONMENT REPORT -- record this Colab session's environment (Python, packages, GPU,
#    driver) to a local JSON file for debugging. Each training run also stores its own copy
#    inside run_record.json, so this file is only a session snapshot and is not kept.
# ---------------------------------------------------------------------------------------------
sh(sys.executable, "scripts/env_report.py", "--out", "/content/env_report_session.json")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 9. RAW BIOLOGY DATA -- rebuild the raw data root locally from the shared ZIP (D-014).
#    E004 still needs it: labels, sessions and the split always come from the raw data; only
#    the images are swapped for the enhanced ones (D-017). The layout must match the raw
#    fingerprint exactly, which cell 11 checks. zExtra is copied only so the raw root is
#    complete; it never reaches training (excluded in the frozen dataset). The shared ZIP has
#    no "Lobster AI Detection Training Data.docx"; the training-subset check does not need it.
#    Everything is written under /content (this machine); nothing is written back to Drive.
# ---------------------------------------------------------------------------------------------
import shutil, zipfile
from pathlib import Path

ZIP_ID = "1_97E8IGX_euJLAzXlyfpukhvRjg2XhfM"          # shared Biology ZIP on Google Drive (D-014)
ZIP_PATH = Path("/content/biology_dataset.zip")
OUTER = Path("/content/biology_outer")                # the outer ZIP unpacked, before rearranging
RAW = Path(LOCAL_RAW)

# Start clean, so a re-run never mixes old and new files.
for d in (RAW, OUTER):
    if d.exists():
        shutil.rmtree(d)
RAW.mkdir(parents=True)
OUTER.mkdir()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
print("Downloading Biology ZIP...")
subprocess.run([sys.executable, "-m", "gdown", f"https://drive.google.com/uc?id={ZIP_ID}",
                "-O", str(ZIP_PATH)], check=True)
print("downloaded:", round(ZIP_PATH.stat().st_size / 1e9, 2), "GB")

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(OUTER)
ROOT = OUTER / "AI Lobster Training Dataset"

# Set 100: plain folder.
shutil.copytree(ROOT / "Set 100", RAW / "Set 100")
# Set 2000: images and labels are inner ZIPs; unpack each into its own folder.
for zip_name, folder in [("batch_2 (2k - imgOnly).zip", "batch_2 (2k - imgOnly)"),
                         ("batch_2_2k_labels.zip", "batch_2_2k_labels")]:
    dst = RAW / "Set 2000" / folder
    dst.mkdir(parents=True)
    with zipfile.ZipFile(ROOT / "Set 2000" / zip_name) as z:
        z.extractall(dst)
# zExtra: copied so the raw root is complete (never trained on).
shutil.copytree(ROOT / "zExtra Photos", RAW / "zExtra Photos")
# Project documents that are part of the raw root (when present in the ZIP).
for name in ["Lobster AI Detection Training Data.docx", "What's been done.xlsx"]:
    if (ROOT / name).exists():
        shutil.copy2(ROOT / name, RAW / name)

# Every later project script finds the raw data through this variable.
os.environ["UWCT_RAW_ROOT"] = LOCAL_RAW
print("raw root:", LOCAL_RAW, "->", sorted(p.name for p in RAW.iterdir()))
for sub in ("Set 100", "Set 2000", "zExtra Photos"):
    assert (RAW / sub).is_dir(), f"missing {sub} in the raw root"
print("Cell 9 complete.")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 10. ENHANCED IMAGES (D-017) -- the one thing that differs from E001.
#     a. Read the dataset version from the E004 config and confirm it uses the enhanced source.
#     b. Copy Harsh's ZIP from Drive to local disk (much faster to read than the Drive mount).
#     c. Unpack only images/<folder>/<name>.png with the repo script; macOS junk is skipped.
#     The folders inside Harsh's ZIP are NOT used as a split: dataset v2 finds each image by its
#     filename and keeps the v1 split and labels.
# ---------------------------------------------------------------------------------------------
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from training.experiment import load_config
from dataset_audit.versioning import image_source_of

cfg = load_config(os.path.join(REPO_DIR, EXPERIMENT_CONFIG))
assert cfg["id"] == EXPERIMENT_ID, f"EXPERIMENT_ID {EXPERIMENT_ID} != config id {cfg['id']}"
DATASET_VERSION = cfg["dataset_version"]
IMAGE_SOURCE = image_source_of(DATASET_VERSION)
print("experiment:", EXPERIMENT_ID, "| dataset:", DATASET_VERSION, "| image source:", IMAGE_SOURCE)
assert DATASET_VERSION == "v2" and IMAGE_SOURCE == "enhanced_2026-09", "E004 must use dataset v2 (enhanced)"

LOCAL_ZIP = "/content/enhanced_2026-09.zip"
if os.path.isdir(LOCAL_ENHANCED):
    shutil.rmtree(LOCAL_ENHANCED)          # start clean, like cell 9
print("copying Harsh's ZIP from Drive...")
shutil.copy(ENHANCED_ZIP, LOCAL_ZIP)
sh(sys.executable, "scripts/unpack_image_source.py", "--zip", LOCAL_ZIP, "--out", LOCAL_ENHANCED)

# Every later project script finds the enhanced images through this variable.
os.environ["UWCT_ENHANCED_ROOT"] = LOCAL_ENHANCED
print("Cell 10 complete.")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 11. RAW CHECKS -- prove the local copies are exactly the recorded raw data before building
#     anything from them. Both checks must end in MATCH; the script exits with an error otherwise
#     (and run() then stops the notebook).
#     a. Biology (labels + split source): structural check of the training folders
#        (every path + size; .txt label contents hashed) against the D-011 fingerprint.
#     b. Enhanced images: FULL content check -- every PNG is hashed (sha256) and compared with
#        raw_fingerprint_enhanced_2026-09.json, so a different or re-saved delivery cannot
#        pass (D-017). Hashing ~6.8 GB takes under a minute.
# ---------------------------------------------------------------------------------------------
run("raw_check_biology", sys.executable, "scripts/check_raw_unchanged.py",
    "--root", LOCAL_RAW, "--subset", "training")

run("raw_check_enhanced", sys.executable, "scripts/check_raw_unchanged.py",
    "--root", LOCAL_ENHANCED, "--full",
    "--fingerprint", "data/processed/raw_fingerprint_enhanced_2026-09.json")

step("both raw checks passed")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 12. BUILD DATASET v2 -- create the YOLO view data/yolo/lobster_v2/ from the frozen v2 manifests.
#     The script first verifies v2 against FROZEN.json (refuses if any manifest changed), then:
#     - labels: from the raw Biology data (same as E001);
#     - images: the enhanced PNGs, found by filename, each checked to have the same size as its
#       raw original; their sha256 goes into build_manifest.csv, so the dataset fingerprint
#       identifies the image content (D-017);
#     - split: v1's session split, unchanged; two data files are written, and the training one
#       has NO test split (CLAUDE.md rule 7).
#     Images are linked, not copied; the raw folders are never written to.
# ---------------------------------------------------------------------------------------------
run("prepare_yolo_v2", sys.executable, "scripts/prepare_yolo_dataset.py", "--version", DATASET_VERSION)
step(f"dataset built: data/yolo/lobster_{DATASET_VERSION}")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 13. VERIFY DATASET v2 -- before any GPU time is spent:
#     a. compact summary from build_summary.json; the dataset fingerprint must equal the one
#        committed in the E004 config (it identifies labels, split AND image content);
#     b. Ultralytics' own label scanner (the code path training uses) must see exactly the
#        images, backgrounds and boxes the build wrote, per split. Must end in VERIFIED.
#     Counting test labels here is not an evaluation; the training data file has no test split.
# ---------------------------------------------------------------------------------------------
import json

DATASET_DIR = os.path.join(REPO_DIR, "data", "yolo", f"lobster_{DATASET_VERSION}")
summary = json.load(open(os.path.join(DATASET_DIR, "build_summary.json"), encoding="utf-8"))

fp = summary["dataset_fingerprint"]
step(f"dataset fingerprint: {fp[:12]}...  (config: {cfg['dataset_fingerprint'][:12]}...)")
assert fp == cfg["dataset_fingerprint"], "dataset fingerprint differs from the E004 config -- STOP"

for split, c in summary["splits"].items():
    print(f"    {split:5}  images={c.get('images')}  positive={c.get('positive')}  "
          f"negative={c.get('negative', 0)}  boxes={c.get('instances')}")

run("verify_yolo_v2", sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", DATASET_DIR)
step("dataset v2 VERIFIED -- ready for preflight")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 14. PREFLIGHT -- run every training check WITHOUT training:
#     config valid and matching the built dataset (fingerprint, class names, counts), training
#     data file has no test split, Ultralytics version = pin, and the exact pretrained checkpoint
#     (yolo26s.pt) downloads and loads. Writes nothing to the runs folder.
#     If yolo26s.pt is unavailable in ultralytics 8.4.155 this STOPS: report it, do NOT change
#     the model (it would break the comparison with E001).
#     TRAIN is reused by the smoke, full and resume cells below.
# ---------------------------------------------------------------------------------------------
TRAIN = [sys.executable, "scripts/train_yolo.py", "--config", EXPERIMENT_CONFIG,
         "--runs-root", RUNS_ROOT, "--weights-dir", WEIGHTS_DIR]

run("preflight_E004", *TRAIN, "--preflight-only", tail=20)
step("preflight passed -- ready for the smoke test")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 15. SMOKE TEST -- 1 epoch on 10% of train, validated on val. Proves the whole training path
#     (GPU, data loading, checkpoint writing to Drive, run record) works before the ~3 h full
#     run. Output: RUNS_ROOT/E004_smoke_<utc>/ (separate from the real E004 folder).
#     Smoke metrics mean nothing about model quality -- never compare or report them.
#     Enabled only on purpose: set RUN_SMOKE = True.
# ---------------------------------------------------------------------------------------------
import glob

RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled -- set RUN_SMOKE = True"

run("smoke_E004", *TRAIN, "--smoke", tail=20)

# Show where the smoke run was written and that its record exists.
smoke_dirs = sorted(glob.glob(os.path.join(RUNS_ROOT, f"{EXPERIMENT_ID}_smoke_*")))
assert smoke_dirs, "no smoke run folder found on Drive"
latest = smoke_dirs[-1]
step(f"smoke run folder: {latest}")
step("contents: " + ", ".join(sorted(os.listdir(latest))))
step("smoke test passed -- review before enabling the full run")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 16. REVIEW BEFORE THE FULL RUN -- read-only.
#     a. The smoke run record must identify E004 correctly: commit, dataset v2 fingerprint,
#        seed, model checkpoint, splits used for training = train + val only.
#     b. E001's recorded training time: a planning estimate for the full run. It counts only
#        COMPLETED launches (interrupted ones have no end time), so after resumes it understates
#        the wall-clock time: E001 recorded 0.6 h but took about 3 h with two crashes.
#     Smoke metrics are NOT printed: they say nothing about model quality.
# ---------------------------------------------------------------------------------------------
smoke = json.load(open(os.path.join(latest, "run_record.json"), encoding="utf-8"))
git_info = smoke["git"]
data_info = smoke["dataset"]
model_info = smoke["model"]

print("    kind        ", smoke["kind"])
print("    commit      ", git_info["commit"][:7], "| dirty:", git_info["dirty"])
print("    dataset     ", data_info["version"], data_info["fingerprint"][:12] + "...")
print("    trained on  ", data_info.get("splits_used_for_training"))
print("    seed        ", smoke["seed"])
print("    model       ", model_info["name"], model_info["checkpoint_sha256"][:12] + "...")

assert git_info["commit"] == COMMIT, "smoke run used a different commit"
assert data_info["fingerprint"] == cfg["dataset_fingerprint"], "smoke run used a different dataset"
assert data_info.get("splits_used_for_training") == ["train", "val"], "unexpected splits"

e001 = json.load(open(os.path.join(RUNS_ROOT, "E001", "run_record.json"), encoding="utf-8"))
e001_hours = e001["duration"]["train_seconds_recorded"] / 3600
e001_launches = len(e001["duration"]["launches"])
step(f"E001 full run: {e001_hours:.1f} h recorded, {e001_launches} launch(es), "
     f"{e001.get('epochs_completed')} epochs")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 17. EPOCH LOGGER -- defines train_with_epoch_log(), used by the FULL RUN and RESUME cells.
#     Run this cell before either of them; it starts nothing by itself.
#     Output is kept SHORT on purpose (long output crashed Colab during E001):
#     - Ultralytics' own output (progress bars etc.) goes ONLY to /content/logs/<label>.log;
#     - every `poll_s` seconds, train/results.csv is read and ONE line is printed per finished
#       epoch: time window, duration, val P / R / mAP50 / mAP50-95, train losses, best val
#       mAP50 so far. All metrics are on the VAL split; the test split is never touched.
#     Times are Colab time (UTC). Epoch end times are when the line was seen (+- poll_s).
#     After a resume, "best" also counts the epochs from before the disconnect.
# ---------------------------------------------------------------------------------------------
import csv

RUN_DIR = os.path.join(RUNS_ROOT, EXPERIMENT_ID)
RESULTS_CSV = os.path.join(RUN_DIR, "train", "results.csv")
EPOCHS = cfg["train_args"]["epochs"]


def read_results(path):
    """Rows of Ultralytics' results.csv with stripped keys; [] if not written yet."""
    try:
        with open(path, newline="") as f:
            rows = list(csv.DictReader(f))
    except OSError:
        return []
    return [{k.strip(): (v or "").strip() for k, v in r.items() if k} for r in rows]


def num(row, key):
    """Float value of a results.csv column, or None if missing/unparseable."""
    try:
        return float(row.get(key, ""))
    except ValueError:
        return None


def fmt(x, nd=3):
    return "  -  " if x is None else f"{x:.{nd}f}"


def train_with_epoch_log(label, *cmd, poll_s=30, tail=15):
    """Run training silently (full output -> log) and print one line per finished epoch."""
    log_path = os.path.join(LOG_DIR, re.sub(r"[^A-Za-z0-9_.-]+", "_", label) + ".log")
    earlier = read_results(RESULTS_CSV)  # non-empty only when resuming
    seen = len(earlier)
    best = (None, None)  # (val mAP50, epoch), including epochs from before a resume
    for i, row in enumerate(earlier):
        m = num(row, "metrics/mAP50(B)")
        if m is not None and (best[0] is None or m > best[0]):
            best = (m, int(num(row, "epoch") or i + 1))
    step(f"START {label} -- one line per epoch; full output: {log_path}")
    if seen:
        step(f"results.csv already has {seen} epoch(s); best so far mAP50 {fmt(best[0])} @ep{best[1]}")
    t_start = time.time()
    last_mark = t_start
    with open(log_path, "a") as log:
        p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=log,
                             stderr=subprocess.STDOUT, text=True)
        try:
            while True:
                code = p.poll()
                rows = read_results(RESULTS_CSV)
                for i in range(seen, len(rows)):
                    row = rows[i]
                    m50 = num(row, "metrics/mAP50(B)")
                    if m50 is None:  # row still being written; read it next time
                        break
                    now = time.time()
                    t_cum = num(row, "time")  # cumulative seconds, if this version writes it
                    t_prev = num(rows[i - 1], "time") if i > 0 else 0.0
                    secs = None
                    if t_cum is not None and t_prev is not None:
                        secs = t_cum - t_prev
                    if secs is None or secs <= 0:  # e.g. the time column restarts after a resume
                        secs = now - last_mark
                    last_mark = now
                    ep = int(num(row, "epoch") or i + 1)
                    if best[0] is None or m50 > best[0]:
                        best = (m50, ep)
                    t_end = time.strftime("%H:%M:%S")
                    t_beg = time.strftime("%H:%M:%S", time.gmtime(now - secs))
                    print(f"ep {ep:3d}/{EPOCHS}  {t_beg}->{t_end} ({secs:4.0f}s)  "
                          f"P {fmt(num(row, 'metrics/precision(B)'))}  "
                          f"R {fmt(num(row, 'metrics/recall(B)'))}  "
                          f"mAP50 {fmt(m50)}  "
                          f"mAP50-95 {fmt(num(row, 'metrics/mAP50-95(B)'))}  "
                          f"box {fmt(num(row, 'train/box_loss'), 2)}  "
                          f"cls {fmt(num(row, 'train/cls_loss'), 2)}  "
                          f"| best mAP50 {fmt(best[0])} @ep{best[1]}", flush=True)
                    seen = i + 1
                if code is not None:
                    break
                time.sleep(poll_s)
        except KeyboardInterrupt:
            p.terminate()  # checkpoints on Drive stay; continue with the RESUME cell
            step("stopped by user -- resume with the RESUME cell")
            raise
    with open(log_path) as f:  # short, meaningful tail of the log
        lines = [l.rstrip() for l in f if l.strip() and not PROGRESS_RE.search(l)]
    for line in lines[-tail:]:
        print("    " + line[:200])
    hours = (time.time() - t_start) / 3600
    if code != 0:
        raise RuntimeError(f"FAILED {label} (exit {code}) after {hours:.2f} h -- log: {log_path}")
    step(f"DONE  {label} ({hours:.2f} h, {seen} epochs in results.csv)")


step("epoch logger ready")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 18. FULL TRAINING RUN -> RUNS_ROOT/E004/ (checkpoints on Drive after every epoch).
#     About 2 minutes per epoch on a T4; early stopping (patience 30) may end it before 100.
#     Refuses to start if RUNS_ROOT/E004/ already exists (use RESUME instead).
#     Keep the tab open; files updated on Drive while it runs: E004/train/results.csv (one row
#     per epoch), E004/train/weights/last.pt (every epoch) and best.pt (when val improves).
#     Enabled only on purpose: set RUN_FULL = True, after the smoke test was reviewed.
# ---------------------------------------------------------------------------------------------
RUN_FULL = False  # set to True deliberately
assert RUN_FULL, "Full training not enabled -- set RUN_FULL = True"
train_with_epoch_log("full_E004", *TRAIN)

In [ ]:
# ---------------------------------------------------------------------------------------------
# 19. RESUME -- only after the Colab MACHINE was lost, not just the browser connection.
#     "Connecting..." in the browser does NOT mean training stopped: during E004 the machine kept
#     training while the page reconnected. Check MyDrive/uwct/runs/E004/train/results.csv first:
#     if it still gains a row every ~2 minutes, touch nothing and wait. Re-running the setup
#     cells on a machine that is still training would delete the data under it.
#     Only if results.csv stopped growing and E004/run_record.json does not exist:
#       1. reconnect (T4 GPU); keep RUN_SMOKE and RUN_FULL = False;
#       2. select cell 15 -> Runtime -> "Run before" (cells 2-14 rebuild the data at the same
#          paths, which the checkpoint expects);
#       3. run cell 17 (epoch logger), then this cell with RESUME = True.
#     train.py refuses to resume if the commit, config, dataset fingerprint or Ultralytics version
#     differs from preflight.json. Training continues from E004/train/weights/last.pt.
# ---------------------------------------------------------------------------------------------
RESUME = False  # set to True deliberately
assert RESUME, "Resume not enabled -- set RESUME = True"
train_with_epoch_log("resume_E004", *TRAIN, "--resume")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 20. RUN SUMMARY -- read-only. Shows the facts the run record holds, so they can be checked
#     before the record is registered on the local PC (scripts/register_run.py).
#     All metrics here are on the VAL split (Ultralytics val defaults, conf 0.001, IoU 0.7).
# ---------------------------------------------------------------------------------------------
rec = json.load(open(os.path.join(RUN_DIR, "run_record.json"), encoding="utf-8"))

print("    run           ", rec["run_name"], "| kind:", rec["kind"])
print("    commit        ", rec["git"]["commit"][:7], "| dirty:", rec["git"]["dirty"])
print("    dataset       ", rec["dataset"]["version"], rec["dataset"]["fingerprint"][:12] + "...")
print("    trained on    ", rec["dataset"].get("splits_used_for_training"))
print("    model         ", rec["model"]["name"], rec["model"]["checkpoint_sha256"][:12] + "...")
print("    seed          ", rec["seed"], "| epochs completed:", rec.get("epochs_completed"))

dur = rec["duration"]
print("    launches      ", len(dur["launches"]),
      "| recorded train time: %.2f h" % (dur["train_seconds_recorded"] / 3600))
for launch in dur["launches"]:
    print("        ", launch["mode"], launch["started_utc"], "->", launch["ended_utc"],
          "|", launch["outcome"])

print("    best.pt sha256", rec["outputs"]["best"]["sha256"][:12] + "...")
print("    val metrics   ", json.dumps(rec["val_metrics"]))

assert rec["git"]["commit"] == COMMIT and rec["git"]["dirty"] is False
assert rec["dataset"]["fingerprint"] == cfg["dataset_fingerprint"]
assert rec["dataset"].get("splits_used_for_training") == ["train", "val"]
step("run record checked -- download run_record.json and train/results.csv for registration")

## After the run (local PC)

1. Download `MyDrive/uwct/runs/E004/run_record.json` and `MyDrive/uwct/runs/E004/train/results.csv`
   into `experiments/records/E004/` (D-013), and check their sha256 against the record.
2. Register: `python scripts/register_run.py experiments/records/E004/run_record.json`.
3. Update the run log in `docs/EXPERIMENTS.md` by hand.

The E004 **test evaluation** is not part of this notebook. It uses the FINAL TEST cells of
`notebooks/colab_train_baseline.ipynb`, and only after E004 is declared in
`experiments/final_candidates.csv`, which waits for the review-list check in D-018.